# MLOps Unit-II Practical: ML Monitoring, Logging & Error Tracking

**Dataset required:** `kidney_disease_cleaned.csv`  
**Purpose:** Classroom demonstration only. This is not a medical diagnostic system.

## Tasks covered
1. Load and inspect data
2. Prepare data and train a Random Forest classifier
3. Track Accuracy, Precision, Recall, F1-score
4. Measure batch prediction latency
5. Check missing values and unseen categories
6. Simulate numeric feature drift and calculate mean-shift scores
7. Compare baseline and shifted-data metrics
8. Save CSV reports
9. Demonstrate application logging and exception tracking

**Before running:** Put `kidney_disease_cleaned.csv` beside this notebook. Run cells in order.

In [ ]:
%pip install pandas numpy scikit-learn matplotlib

In [ ]:
import time
import logging
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

DATASET_PATH = Path("kidney_disease_cleaned.csv")
REPORT_DIR = Path("monitoring_reports")
LOG_DIR = Path("logs")
REPORT_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Download/provide kidney_disease_cleaned.csv "
        "and place it in the same folder as this notebook."
    )

df = pd.read_csv(DATASET_PATH)
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

print("Dataset shape:", df.shape)
display(df.head())
print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).head(15))
print("\nTarget class counts:")
print(df["classification"].value_counts(dropna=False))

## Task 1 — Normalize target, split data, and train the model

In [ ]:
target = (
    df["classification"].astype("string").str.lower().str.strip()
    .str.replace(r"\.$", "", regex=True)
    .str.replace(r"\s+", "", regex=True)
)
label_map = {"ckd": "ckd", "notckd": "notckd", "not_ckd": "notckd"}
df["classification"] = target.map(label_map)
df = df.dropna(subset=["classification"]).copy()

X = df.drop(columns=["classification", "id"], errors="ignore")
y = df["classification"]
if y.nunique() < 2:
    raise ValueError("Expected at least two target classes in 'classification'.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

transformers = []
if numeric_features:
    transformers.append(("num", SimpleImputer(strategy="median"), numeric_features))
if categorical_features:
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
    transformers.append(("cat", cat_pipeline, categorical_features))

preprocessor = ColumnTransformer(transformers)
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight="balanced"
    ))
])
model.fit(X_train, y_train)
print(f"Model trained. Training rows: {len(X_train)}; test rows: {len(X_test)}")

## Task 2 — Model performance metrics

In [ ]:
baseline_pred = model.predict(X_test)
baseline_metrics = {
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision_macro": precision_score(y_test, baseline_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, baseline_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, baseline_pred, average="macro", zero_division=0),
}
display(pd.Series(baseline_metrics, name="Score").round(4))
print(classification_report(y_test, baseline_pred, zero_division=0))
print("Confusion matrix (class order:", list(model.classes_), ")")
display(pd.DataFrame(
    confusion_matrix(y_test, baseline_pred, labels=model.classes_),
    index=[f"Actual {c}" for c in model.classes_],
    columns=[f"Predicted {c}" for c in model.classes_]
))

## Task 3 — Measure batch prediction latency

In [ ]:
batch = X_test.copy()
latencies_ms = []
for _ in range(10):
    start = time.perf_counter()
    model.predict(batch)
    latencies_ms.append((time.perf_counter() - start) * 1000)

latency_summary = {
    "batch_size": len(batch),
    "mean_batch_latency_ms": float(np.mean(latencies_ms)),
    "median_batch_latency_ms": float(np.median(latencies_ms)),
    "max_batch_latency_ms": float(np.max(latencies_ms)),
    "mean_per_record_ms": float(np.mean(latencies_ms) / max(len(batch), 1)),
}
display(pd.Series(latency_summary).round(3))
print("Note: this is notebook batch-inference latency, not API response latency.")

## Task 4 — Data-quality checks and unseen categories

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": X_test.isna().sum(),
    "missing_percent": X_test.isna().mean() * 100
}).sort_values("missing_count", ascending=False)
display(missing_report)

unseen_categories = {}
for col in categorical_features:
    train_values = set(X_train[col].dropna().astype(str).unique())
    test_values = set(X_test[col].dropna().astype(str).unique())
    unseen = sorted(test_values - train_values)
    if unseen:
        unseen_categories[col] = unseen
print("Unseen categories:", unseen_categories or "None found")

## Task 5 — Simulate data drift and calculate mean-shift scores

In [ ]:
reference_data = X_test.copy()
drifted_data = X_test.copy()
shifted_feature = None
shift_amount = 0.0

if numeric_features:
    shifted_feature = numeric_features[0]
    ref_values = pd.to_numeric(reference_data[shifted_feature], errors="coerce")
    ref_median = ref_values.median()
    shift_amount = 0.5 * abs(ref_median) if pd.notna(ref_median) and ref_median != 0 else 1.0
    drifted_data[shifted_feature] = (
        pd.to_numeric(drifted_data[shifted_feature], errors="coerce") + shift_amount
    )
    print("Artificially shifted feature:", shifted_feature)
    print("Shift added:", shift_amount)
else:
    print("No numeric features available for drift simulation.")

drift_rows = []
for col in numeric_features:
    ref = pd.to_numeric(reference_data[col], errors="coerce")
    prod = pd.to_numeric(drifted_data[col], errors="coerce")
    ref_std = ref.std()
    score = abs(prod.mean() - ref.mean()) / ref_std if pd.notna(ref_std) and ref_std > 0 else np.nan
    drift_rows.append({
        "feature": col,
        "reference_mean": ref.mean(),
        "shifted_mean": prod.mean(),
        "reference_std": ref_std,
        "mean_shift_score": score
    })

drift_report = pd.DataFrame(drift_rows)
if not drift_report.empty:
    drift_report = drift_report.sort_values("mean_shift_score", ascending=False, na_position="last")
display(drift_report)
print("Interpretation: a larger score suggests a larger shift relative to reference variability.")
print("This simple score is a screening statistic, not proof of model degradation.")

## Task 6 — Compare baseline and shifted-data performance

In [ ]:
shifted_pred = model.predict(drifted_data)
reference_pred = model.predict(reference_data)

def metric_row(name, truth, pred):
    return {
        "dataset": name,
        "accuracy": accuracy_score(truth, pred),
        "precision_macro": precision_score(truth, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(truth, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(truth, pred, average="macro", zero_division=0),
    }

comparison = pd.DataFrame([
    metric_row("Reference test data", y_test, reference_pred),
    metric_row("Artificially shifted data", y_test, shifted_pred),
])
display(comparison.round(4))
print("The original test labels are reused only because this is a controlled classroom simulation.")

## Task 7 — Save monitoring reports

In [ ]:
summary = {
    **baseline_metrics,
    **latency_summary,
    "test_records": len(X_test),
    "shifted_feature": shifted_feature or "",
    "shift_amount": shift_amount,
    "max_mean_shift_score": (
        drift_report["mean_shift_score"].max()
        if not drift_report.empty else np.nan
    ),
}
pd.DataFrame([summary]).to_csv(REPORT_DIR / "kidney_monitoring_summary.csv", index=False)
drift_report.to_csv(REPORT_DIR / "kidney_drift_report.csv", index=False)
comparison.to_csv(REPORT_DIR / "kidney_performance_comparison.csv", index=False)
missing_report.to_csv(REPORT_DIR / "kidney_missing_values_report.csv")
print("Reports saved in:", REPORT_DIR.resolve())
print("\nFiles:")
for p in sorted(REPORT_DIR.glob("*.csv")):
    print("-", p.name)

## Task 8 — Logging and error tracking

In [ ]:
# Configure a log file and console output.
logger = logging.getLogger("mlops_monitoring")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_DIR / "mlops_monitoring.log", encoding="utf-8")
file_handler.setLevel(logging.INFO)
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

logger.info("Monitoring practical started")
logger.info("Model training completed; model_version=classroom-rf-v1")
logger.info("Baseline metrics: %s", baseline_metrics)
logger.info("Latency summary: %s", latency_summary)
logger.info("Drift simulation feature=%s shift_amount=%s", shifted_feature, shift_amount)

# Demonstrate capturing an exception without crashing the notebook.
try:
    _ = df["this_column_does_not_exist"]
except KeyError:
    logger.exception("Example error tracking: requested column was missing")

print("Log file written to:", (LOG_DIR / "mlops_monitoring.log").resolve())
print("Logging captures timestamps, severity, events, metrics, and exception details.")
print("Avoid logging patient identifiers or raw sensitive records.")

## Conclusion / Result

This practical demonstrates a basic ML monitoring workflow: model quality metrics, batch latency, data-quality checks, a controlled feature-drift simulation, performance comparison, CSV reports, and application logs.

**Important limitations**
- Results are only valid for the supplied dataset and this train/test split.
- Data drift alone does not prove model performance has declined.
- In real production, performance metrics require reliable ground-truth labels.
- The shifted-data comparison reuses test labels only for this artificial classroom exercise.
- This is not a medical diagnostic system.
